# Notebook 01 — Exploratory Data Analysis (EDA)

**Goal:** Load the FI-2010 LOB dataset, understand its structure, and visualize key properties.

**What we learn:**
- Shape of the data (samples x features)
- 10-level order book snapshot structure
- Label distributions across 5 prediction horizons
- Price and volume distributions
- Correlation structure

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.utils import set_seed
from src.data_loader import load_fi2010_raw, prepare_labels, get_lob_column_names

set_seed(42)
sns.set_theme(style='whitegrid', font_scale=1.2)
%matplotlib inline

RESULTS_DIR = Path('../results/plots')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Setup complete.')

## 1. Load the Dataset

In [ ]:
X, y = load_fi2010_raw(data_dir='../data/raw', normalization='zscore')
print(f'Feature matrix X shape: {X.shape}')
print(f'Label matrix y shape:   {y.shape}')
print(f'X dtype: {X.dtype}, y dtype: {y.dtype}')
print(f'X memory: {X.nbytes / 1e6:.1f} MB')

## 2. Understand the LOB Snapshot Structure

Each snapshot contains 40 raw LOB columns (10 levels x 4 values each):
- `ask_price_i`, `ask_vol_i`, `bid_price_i`, `bid_vol_i` for i = 1..10

Plus additional hand-crafted features from the FI-2010 authors (up to 144 total).

In [ ]:
col_names = get_lob_column_names(10)
print(f'Raw LOB columns ({len(col_names)}):')
for i in range(0, len(col_names), 4):
    print(f'  Level {i//4 + 1}: {col_names[i:i+4]}')

# Create a DataFrame of the first 40 columns (raw LOB)
df_lob = pd.DataFrame(X[:, :40], columns=col_names)
print(f'\nLOB DataFrame shape: {df_lob.shape}')
df_lob.head(10)

## 3. Summary Statistics

In [ ]:
print('=== Summary Statistics (first 40 raw LOB columns) ===')
df_lob.describe().round(4)

## 4. Label Distribution Across Horizons

FI-2010 provides labels for 5 prediction horizons:
- k=1 (10 events), k=2 (20), k=3 (30), k=5 (50), k=10 (100)

In [ ]:
horizon_names = ['k=1 (10 events)', 'k=2 (20 events)', 'k=3 (30 events)',
                 'k=5 (50 events)', 'k=10 (100 events)']

fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)
for i, (ax, name) in enumerate(zip(axes, horizon_names)):
    labels_i = prepare_labels(y, horizon=i)
    counts = np.bincount(labels_i, minlength=3)
    bars = ax.bar(['Down', 'Stat.', 'Up'], counts,
                  color=['#e74c3c', '#95a5a6', '#2ecc71'])
    ax.set_title(name, fontsize=11)
    ax.set_ylabel('Count' if i == 0 else '')
    for bar, c in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{c:,}', ha='center', va='bottom', fontsize=9)

fig.suptitle('Label Distribution Across Prediction Horizons', fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/plots/label_distribution.png')

## 5. Mid-Price Time Series

In [ ]:
ask1 = X[:, 0]  # ask_price_1
bid1 = X[:, 2]  # bid_price_1
mid_price = (ask1 + bid1) / 2.0

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Full series
axes[0].plot(mid_price, linewidth=0.5, color='#2c3e50')
axes[0].set_title('Mid-Price — Full Series')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Mid-Price')

# Zoomed (first 2000 steps)
axes[1].plot(mid_price[:2000], linewidth=1.0, color='#2980b9')
axes[1].set_title('Mid-Price — First 2,000 Steps (Zoomed)')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Mid-Price')

plt.tight_layout()
fig.savefig(RESULTS_DIR / 'mid_price_series.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Bid-Ask Spread Distribution

In [ ]:
spread = ask1 - bid1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(spread, bins=100, color='#e67e22', edgecolor='white', alpha=0.8)
axes[0].set_title('Bid-Ask Spread Distribution')
axes[0].set_xlabel('Spread')
axes[0].set_ylabel('Frequency')
axes[0].axvline(np.median(spread), color='red', linestyle='--', label=f'Median={np.median(spread):.4f}')
axes[0].legend()

axes[1].plot(spread[:5000], linewidth=0.5, color='#e67e22')
axes[1].set_title('Spread Over Time (First 5,000 Steps)')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Spread')

plt.tight_layout()
fig.savefig(RESULTS_DIR / 'spread_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Volume at Top Levels

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ask_vols_mean = [X[:, 1 + 4*i].mean() for i in range(10)]
bid_vols_mean = [X[:, 3 + 4*i].mean() for i in range(10)]

x = np.arange(1, 11)
width = 0.35
ax.bar(x - width/2, ask_vols_mean, width, label='Ask Volume', color='#e74c3c', alpha=0.8)
ax.bar(x + width/2, bid_vols_mean, width, label='Bid Volume', color='#2ecc71', alpha=0.8)
ax.set_xlabel('Price Level')
ax.set_ylabel('Mean Volume')
ax.set_title('Average Volume by Price Level')
ax.set_xticks(x)
ax.legend()

plt.tight_layout()
fig.savefig(RESULTS_DIR / 'volume_by_level.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Correlation Heatmap (Raw LOB Features)

In [ ]:
# Only first 20 columns for readability
corr = df_lob.iloc[:, :20].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap='RdBu_r', center=0, annot=False,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap — First 20 LOB Features')
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. LOB Depth Snapshot Visualization

In [ ]:
# Take a single snapshot (row 500)
snapshot_idx = 500
snap = X[snapshot_idx, :40]

ask_prices = snap[0::4][:10]
ask_volumes = snap[1::4][:10]
bid_prices = snap[2::4][:10]
bid_volumes = snap[3::4][:10]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(10), -ask_volumes[::-1], color='#e74c3c', alpha=0.7, label='Ask')
ax.barh(range(10, 20), bid_volumes, color='#2ecc71', alpha=0.7, label='Bid')

ytick_labels = [f'A{10-i} ({ask_prices[9-i]:.2f})' for i in range(10)] + \
               [f'B{i+1} ({bid_prices[i]:.2f})' for i in range(10)]
ax.set_yticks(range(20))
ax.set_yticklabels(ytick_labels, fontsize=9)
ax.set_xlabel('Volume')
ax.set_title(f'LOB Depth Snapshot (t={snapshot_idx})')
ax.legend()
ax.axhline(9.5, color='black', linewidth=2, linestyle='--', alpha=0.5)
ax.text(0, 9.5, ' SPREAD ', ha='center', va='center',
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))

plt.tight_layout()
fig.savefig(RESULTS_DIR / 'lob_depth_snapshot.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA complete! All plots saved to results/plots/')

## Summary

- Dataset has ~150K+ snapshots with 144 features each
- 10-level LOB with ask/bid prices and volumes
- Labels are roughly balanced across 3 classes (varies by horizon)
- Strong correlations between adjacent price levels (as expected)
- Spread is typically tight but varies over time

**Next:** Notebook 02 — Feature Engineering